<a href="https://colab.research.google.com/github/SonLee369/auto-labeling-scripts/blob/main/Bbox_Polygon_Auto_Labeling_Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Hướng dẫn sử dụng Notebook Gắn Nhãn Tự Động này

Notebook này cung cấp giải pháp tự động để tạo các chú thích ban đầu (BBox cho đối tượng, Đa giác cho khu vực có thể lái xe) dựa trên Hướng dẫn Chú thích AI20K v1.0. Nó sử dụng mô hình Mask2Former Panoptic Segmentation (được huấn luyện trên Cityscapes) để xử lý hình ảnh và xuất ra các chú thích ở định dạng XML CVAT 1.1 "cho hình ảnh", sẵn sàng để nhập trực tiếp vào các tác vụ CVAT của bạn.

### Các bước thực hiện:

1.  **Chuẩn bị**:
    *   **Xuất `annotations.xml` từ CVAT**: Trước khi chạy notebook này, hãy xuất một tệp `annotations.xml` *rỗng* từ chính tác vụ CVAT mà bạn dự định tự động gắn nhãn. Tệp này đóng vai trò tham chiếu để đảm bảo ID và đường dẫn hình ảnh chính xác, tránh lỗi nhập. Đặt tệp `annotations.xml` này vào cùng thư mục với notebook này.
    *   **Chuẩn bị thư mục `images/`**: Xuất các hình ảnh từ tác vụ CVAT của bạn và đặt chúng vào một thư mục con có tên `images/` trong cùng thư mục với notebook này. Tập lệnh sẽ khớp các hình ảnh cục bộ này với các tham chiếu từ tệp `annotations.xml`.

2.  **Chạy Notebook**: Thực thi tất cả các ô trong notebook.
    *   Tập lệnh sẽ cài đặt các thư viện cần thiết, tải mô hình Mask2Former và sau đó xử lý từng hình ảnh.
    *   Nó sẽ tạo hộp giới hạn cho các lớp `thing` đã biết (ví dụ: người đi bộ, ô tô, xe tải) và đa giác cho `area/drivable` (từ phân đoạn đường).
    *   Nó cũng sẽ cố gắng xác định `traffic light` và `traffic sign` bằng cách sử dụng các thành phần liên thông, nhưng những điều này yêu cầu xem xét thủ công do những hạn chế của mô hình.

3.  **Tệp đầu ra**: Notebook sẽ tạo hai tệp đầu ra chính trong thư mục `cvat_autolabel_output`:
    *   `annotations.xml`: Tệp này chứa các chú thích đã tạo ở định dạng CVAT 1.1. Bạn sẽ nhập tệp này vào tác vụ CVAT của mình.
    *   `review_log.csv`: Tệp CSV này liệt kê tất cả các trường hợp mà mô hình có độ tin cậy thấp, xác định các vấn đề tiềm ẩn (ví dụ: đối tượng bị che khuất, đối tượng nhỏ) hoặc khi yêu cầu chú thích thủ công rõ ràng (ví dụ: vạch kẻ đường, `area/alternative`).

4.  **Nhập vào CVAT**:
    *   Trong tác vụ CVAT của bạn, hãy chuyển đến "Tasks" -> "[Tên tác vụ của bạn]" -> "Actions" -> "Upload annotations".
    *   Chọn "CVAT 1.1" làm định dạng và tải lên tệp `annotations.xml` do tập lệnh này tạo.

5.  **Xem xét và Tinh chỉnh**:
    *   **Ưu tiên `review_log.csv`**: Cẩn thận kiểm tra `review_log.csv`. Đối với mỗi mục nhập, hãy tạo một "Vấn đề" hoặc một nhận xét trong CVAT, hướng dẫn người chú thích/người đánh giá giải quyết các mối quan tâm cụ thể.
    *   **Chú thích thủ công**: Chú thích thủ công vạch kẻ đường `lane` (polyline) và phân biệt `area/alternative` với `area/drivable` vì mô hình không thể tự động tạo ra những thứ này một cách chính xác.
    *   **Kiểm tra thuộc tính**: Xác nhận các thuộc tính `occluded` và `truncated`, đặc biệt đối với các đối tượng được gắn cờ trong `review_log.csv`.

### Những cân nhắc quan trọng:

*   **Tuân thủ Phân loại**: Tập lệnh này tuân thủ nghiêm ngặt 19 nhãn được phép trong hướng dẫn. Nó cố tình *tránh* tự động gắn nhãn các lớp bổ sung (ví dụ: vỉa hè, tòa nhà, người-như-một-vật) có thể có trong `Labels.md` CVAT rộng hơn nhưng không có trong hướng dẫn chính thức.
*   **Vạch kẻ đường (Polyline)**: Mô hình Mask2Former không cung cấp phân đoạn vạch kẻ đường. Tất cả các chú thích `lane/*` phải được thực hiện thủ công.
*   **`traffic light` / `traffic sign`**: Đây là các lớp "stuff" trong Cityscapes. Tập lệnh sử dụng các thành phần liên thông để tách chúng, nhưng điều này kém tin cậy hơn so với phân đoạn đối tượng. Xem xét kỹ các chú thích này.
*   **`area/alternative`**: Mô hình không thể phân biệt giữa `area/drivable` và `area/alternative`. Tất cả các đoạn đường được gắn nhãn là `area/drivable`; `area/alternative` phải được chú thích thủ công nếu có.
*   **Ngưỡng**: Các ngưỡng độ tin cậy và diện tích tối thiểu (`SCORE_THRESHOLD`, `MIN_BOX_AREA_PX`, v.v.) là các tham số có thể cấu hình. Điều chỉnh chúng dựa trên đặc điểm của tập dữ liệu của bạn và chất lượng chú thích yêu cầu.

In [ ]:
# =============================================================================
# AUTO-LABELING SCRIPT — BBox (object) / Polygon (drivable area) / Polyline (lane)
# Tuân thủ: AI20K Annotation Guideline v1.0 (rule set G01)
#
# Output: CVAT 1.1 "for images" XML -> import THẲNG vào job CVAT như annotation
# (không phải segmentation mask), sau đó bạn review/sửa (bước "edit label").
#
# ĐỌC PHẦN "GHI CHÚ QUAN TRỌNG" Ở CUỐI FILE trước khi chạy thật.
# =============================================================================

# -----------------------------------------------------------------------------
# 1. CÀI ĐẶT THƯ VIỆN & IMPORT
# -----------------------------------------------------------------------------
!pip install -q transformers torch torchvision Pillow numpy opencv-python

from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation
from PIL import Image
import torch
import numpy as np
import cv2
import os
import glob
import shutil
import zipfile
import csv
from xml.etree import ElementTree as ET
from xml.dom import minidom

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng phần cứng: {device} (Khuyên dùng T4 GPU trên Colab)")

# -----------------------------------------------------------------------------
# 2. TAXONOMY ĐÚNG THEO GUIDELINE (CHỈ 19 LABEL ĐƯỢC PHÉP)
#    -> KHÔNG dùng các label thừa trong Labels.md (road, sidewalk, building,
#       wall, fence, pole, vegetation, terrain, sky, person, traffic_light,
#       traffic_sign...) vì guideline mục 1 quy định:
#       "Chỉ annotate các class được liệt kê trong tài liệu."
# -----------------------------------------------------------------------------
BBOX_LABELS = {
    # tên guideline : màu hex lấy đúng từ Labels.md (đồng bộ khi import CVAT)
    "pedestrian":    "#80e060",
    "rider":         "#FF0000",
    "car":           "#00008E",
    "truck":         "#000046",
    "bus":           "#003C64",
    "train":         "#005064",
    "motorcycle":    "#0000E6",
    "bicycle":       "#770B20",
    "traffic light": "#d0a000",
    "traffic sign":  "#502080",
}
POLYGON_LABELS = {
    "area/drivable":    "#4a3d3c",
    "area/alternative": "#62c4b2",
}
POLYLINE_LABELS = {  # KHÔNG auto-generate được (xem ghi chú), chỉ khai báo để nhất quán
    "lane/crosswalk":     "#7df28c",
    "lane/double white":  "#d8c413",
    "lane/double yellow": "#ba3dfd",
    "lane/road curb":     "#6e0189",
    "lane/single other":  "#ff1f60",
    "lane/single white":  "#a4faa4",
    "lane/single yellow": "#544df0",
}

# Mapping từ id class Cityscapes (panoptic) -> tên label theo guideline
# (chỉ các "thing class" mới có instance riêng biệt trong Mask2Former Panoptic)
CITYSCAPES_THING_ID_TO_LABEL = {
    11: "pedestrian",   # Cityscapes "person" ~ guideline "pedestrian"
    12: "rider",
    13: "car",
    14: "truck",
    15: "bus",
    16: "train",
    17: "motorcycle",
    18: "bicycle",
}
# "stuff class" liên quan traffic light/sign -> không có instance riêng,
# sẽ tách bằng connected components (độ tin cậy thấp hơn -> luôn đưa review)
CITYSCAPES_STUFF_ID_TO_LABEL = {
    6: "traffic light",
    7: "traffic sign",
}
CITYSCAPES_ROAD_ID = 0  # dùng để tạo polygon area/drivable

# Ngưỡng để quyết định "đủ tự tin" hay phải đưa review (đúng nguyên tắc
# "không đoán; case không rõ phải đưa review" của guideline)
SCORE_THRESHOLD = 0.55          # confidence tối thiểu của model để giữ box
MIN_BOX_AREA_PX = 12 * 12       # object quá nhỏ -> nghi ngờ, đưa review
OCCLUDED_FILL_RATIO = 0.65      # tỉ lệ pixel-mask / bbox-area thấp -> nghi bị che
MIN_ROAD_CONTOUR_AREA = 2000    # bỏ các mảnh road vụn (nhiễu) khi tạo polygon

# -----------------------------------------------------------------------------
# 3. TẢI MODEL — dùng PANOPTIC (không dùng semantic) để có instance riêng
#    cho từng object, phục vụ đúng yêu cầu "mỗi object = một annotation riêng"
# -----------------------------------------------------------------------------
print("Đang tải Mask2Former Panoptic (Cityscapes)...")
ten_mo_hinh = "facebook/mask2former-swin-large-cityscapes-panoptic"
processor = Mask2FormerImageProcessor.from_pretrained(ten_mo_hinh)
model = Mask2FormerForUniversalSegmentation.from_pretrained(ten_mo_hinh).to(device)
model.eval()

# -----------------------------------------------------------------------------
# 4. CHUẨN BỊ THƯ MỤC OUTPUT
# -----------------------------------------------------------------------------
OUTPUT_DIR = "cvat_autolabel_output"
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

review_rows = []  # log các case không chắc -> để reviewer xử lý (mục 5 guideline)


def add_review(image_name, issue_type, detail):
    """issue_type theo đúng 4 loại trong guideline mục 5:
    UNCERTAIN_CLASS / UNCERTAIN_BOUNDARY / UNCERTAIN_SCOPE / ATTRIBUTE_CHECK"""
    review_rows.append({"image": image_name, "issue_type": issue_type, "detail": detail})


# -----------------------------------------------------------------------------
# 5. LẤY DANH SÁCH FRAME CHUẨN TỪ FILE annotations.xml GỐC (export từ CVAT)
#
#    QUAN TRỌNG — lý do bắt buộc làm theo cách này:
#    CVAT so khớp ảnh khi import theo đúng giá trị `name` trong thẻ <image>,
#    và giá trị này thường là ĐƯỜNG DẪN TƯƠNG ĐỐI ĐẦY ĐỦ chứ không chỉ tên
#    file, ví dụ: "w1/bbox_polygon/G05/G05_B026.jpg" (không phải "G05_B026.jpg").
#    Nếu script tự đoán tên theo os.path.basename() của ảnh trong thư mục
#    images/, phần tiền tố thư mục sẽ bị mất -> CVAT báo lỗi
#    "Could not match item id ... with any task frame" khi import.
#
#    Cách chắc chắn nhất: LUÔN xuất một bản annotations.xml TRỐNG (chưa có
#    nhãn) từ chính job/task cần import (Job -> ... -> Export dataset ->
#    "CVAT for images 1.1", không cần tick Save images) và đặt tên file đó
#    là REFERENCE_XML_PATH bên dưới. Script sẽ đọc chính xác id/name/
#    width/height của từng frame từ file này, đảm bảo khớp 100% khi import
#    ngược lại vào CVAT.
# -----------------------------------------------------------------------------
REFERENCE_XML_PATH = "annotations.xml"   # <-- file export rỗng từ job cần auto-label
IMAGE_DIR = "images"                     # thư mục chứa ảnh gốc (có thể lồng thư mục con)

if not os.path.isfile(REFERENCE_XML_PATH):
    raise FileNotFoundError(
        f"Không tìm thấy '{REFERENCE_XML_PATH}'. Hãy export dataset "
        "(format 'CVAT for images 1.1', không cần Save images) từ chính "
        "job bạn muốn auto-label, đặt file annotations.xml đó cùng thư mục "
        "với script rồi chạy lại."
    )


def load_reference_frames(xml_path):
    """Đọc danh sách frame CHUẨN (id, name đầy đủ, width, height) từ file
    annotations.xml gốc do CVAT export -> dùng làm nguồn chân lý duy nhất
    khi ghi lại file kết quả, tránh sai lệch tên/id khi import ngược lại."""
    tree = ET.parse(xml_path)
    frames = []
    for image_el in tree.getroot().findall("image"):
        frames.append({
            "id": image_el.get("id"),
            "name": image_el.get("name"),          # GIỮ NGUYÊN full path, không basename
            "width": int(image_el.get("width")),
            "height": int(image_el.get("height")),
        })
    if not frames:
        raise ValueError(f"'{xml_path}' không chứa thẻ <image> nào để tham chiếu.")
    return frames


def find_local_image(image_dir, ref_name):
    """Tìm file ảnh thực tế trên máy khớp với tên frame chuẩn (so theo
    basename, vì ảnh export ra có thể bị làm phẳng thư mục hoặc giữ
    nguyên cấu trúc con w1/bbox_polygon/G05/...)."""
    target_basename = os.path.basename(ref_name)
    # Ưu tiên đúng path y hệt nếu tồn tại
    exact_path = os.path.join(image_dir, ref_name)
    if os.path.isfile(exact_path):
        return exact_path
    # Fallback: tìm đệ quy theo basename trong toàn bộ image_dir
    matches = glob.glob(os.path.join(image_dir, "**", target_basename), recursive=True)
    return matches[0] if matches else None


reference_frames = load_reference_frames(REFERENCE_XML_PATH)
print(f"Đã đọc {len(reference_frames)} frame chuẩn từ '{REFERENCE_XML_PATH}'.")

missing_local_files = [f for f in reference_frames if find_local_image(IMAGE_DIR, f["name"]) is None]
if missing_local_files:
    names = ", ".join(os.path.basename(f["name"]) for f in missing_local_files[:10])
    raise FileNotFoundError(
        f"Thiếu {len(missing_local_files)} ảnh trong '{IMAGE_DIR}/' so với danh sách "
        f"tham chiếu (vd: {names}...). Hãy export đủ ảnh của đúng job này trước khi chạy."
    )

print(f"Tổng số ảnh cần auto-label: {len(reference_frames)} (khớp 100% với '{REFERENCE_XML_PATH}')")

# -----------------------------------------------------------------------------
# 6. HÀM TIỆN ÍCH: MASK -> POLYGON (contour, đơn giản hoá điểm, không self-intersect)
# -----------------------------------------------------------------------------
def mask_to_polygons(mask_uint8, min_area=MIN_ROAD_CONTOUR_AREA, epsilon_ratio=0.003):
    """Trả về list các polygon (list điểm (x,y)) từ 1 binary mask.
    - Dùng RETR_EXTERNAL để không tạo lỗ / vòng lặp gây self-intersection.
    - approxPolyDP để giảm điểm thừa trên đoạn thẳng (đúng mục 4.1 guideline)."""
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue
        epsilon = epsilon_ratio * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)
        if len(approx) < 3:
            continue
        pts = [(float(p[0][0]), float(p[0][1])) for p in approx]
        polygons.append(pts)
    return polygons


def bbox_touches_border(xtl, ytl, xbr, ybr, w, h, margin=1):
    return xtl <= margin or ytl <= margin or xbr >= (w - margin) or ybr >= (h - margin)


# -----------------------------------------------------------------------------
# 7. XỬ LÝ TỪNG ẢNH
# -----------------------------------------------------------------------------
root = ET.Element("annotations")
ET.SubElement(root, "version").text = "1.1"

for frame in reference_frames:
    img_idx = frame["id"]
    img_name = frame["name"]          # GIỮ NGUYÊN full path để ghi lại vào XML output
    img_path = find_local_image(IMAGE_DIR, img_name)
    print(f"[id={img_idx}] Đang xử lý: {img_name}")

    image = Image.open(img_path).convert("RGB")
    W, H = image.size

    # Cảnh báo nếu kích thước ảnh thực tế khác với kích thước ghi trong file
    # tham chiếu -> có thể ảnh bị resize/nhầm file, cần người kiểm tra lại
    if (W, H) != (frame["width"], frame["height"]):
        add_review(img_name, "UNCERTAIN_SCOPE",
                   f"Kích thước ảnh thực tế ({W}x{H}) khác file tham chiếu "
                   f"({frame['width']}x{frame['height']}) -> kiểm tra lại đúng ảnh")

    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    panoptic_result = processor.post_process_panoptic_segmentation(
        outputs, target_sizes=[(H, W)]
    )[0]
    seg_map = panoptic_result["segmentation"].cpu().numpy()       # HxW, giá trị = segment_id
    segments_info = panoptic_result["segments_info"]              # list dict: id,label_id,score,...

    image_el = ET.SubElement(root, "image", {
        "id": img_idx,
        "name": img_name,
        "width": str(frame["width"]),
        "height": str(frame["height"]),
    })

    has_any_object = False

    # ---- 7.1 BOUNDING BOX cho object instance (thing class) ----
    for seg in segments_info:
        cityscapes_id = seg["label_id"]
        if cityscapes_id not in CITYSCAPES_THING_ID_TO_LABEL:
            continue

        label_name = CITYSCAPES_THING_ID_TO_LABEL[cityscapes_id]
        instance_mask = (seg_map == seg["id"])
        ys, xs = np.where(instance_mask)
        if len(xs) == 0:
            continue

        xtl, xbr = float(xs.min()), float(xs.max())
        ytl, ybr = float(ys.min()), float(ys.max())
        box_area = max(1.0, (xbr - xtl) * (ybr - ytl))
        mask_pixels = int(instance_mask.sum())
        score = float(seg.get("score", 1.0))

        # --- Không đoán khi model không đủ tự tin / object quá nhỏ ---
        if score < SCORE_THRESHOLD:
            add_review(img_name, "UNCERTAIN_CLASS",
                       f"{label_name} score={score:.2f} thấp -> cần người xác nhận class")
            continue
        if box_area < MIN_BOX_AREA_PX:
            add_review(img_name, "UNCERTAIN_SCOPE",
                       f"{label_name} box quá nhỏ ({box_area:.0f}px) -> cần người xác nhận có thuộc scope không")
            continue

        truncated = bbox_touches_border(xtl, ytl, xbr, ybr, W, H)
        fill_ratio = mask_pixels / box_area
        occluded = fill_ratio < OCCLUDED_FILL_RATIO

        # fill_ratio quá thấp bất thường -> nghi ngờ boundary, vẫn giữ nhưng gắn review
        if fill_ratio < 0.35:
            add_review(img_name, "UNCERTAIN_BOUNDARY",
                       f"{label_name} fill_ratio={fill_ratio:.2f} rất thấp -> kiểm tra lại boundary/occlusion")

        box_el = ET.SubElement(image_el, "box", {
            "label": label_name,
            "occluded": "1" if occluded else "0",
            "xtl": f"{xtl:.2f}", "ytl": f"{ytl:.2f}",
            "xbr": f"{xbr:.2f}", "ybr": f"{ybr:.2f}",
            "z_order": "0",
        })
        # Khớp đúng attribute "truncated" đã khai báo trong Labels.md
        attr_el = ET.SubElement(box_el, "attribute", {"name": "truncated"})
        attr_el.text = "true" if truncated else "false"
        has_any_object = True

        if occluded:
            add_review(img_name, "ATTRIBUTE_CHECK",
                       f"{label_name} occluded=true (auto) -> reviewer xác nhận lại")

    # ---- 7.2 BOUNDING BOX cho traffic light / traffic sign (stuff -> connected components) ----
    for cityscapes_id, label_name in CITYSCAPES_STUFF_ID_TO_LABEL.items():
        stuff_mask = (seg_map == next(
            (s["id"] for s in segments_info if s["label_id"] == cityscapes_id), -1
        )).astype(np.uint8)
        if stuff_mask.sum() == 0:
            continue
        num_labels, cc_labels = cv2.connectedComponents(stuff_mask)
        for cc_id in range(1, num_labels):
            ys, xs = np.where(cc_labels == cc_id)
            if len(xs) == 0:
                continue
            box_area = (xs.max() - xs.min()) * (ys.max() - ys.min())
            if box_area < MIN_BOX_AREA_PX:
                continue
            xtl, xbr = float(xs.min()), float(xs.max())
            ytl, ybr = float(ys.min()), float(ys.max())
            truncated = bbox_touches_border(xtl, ytl, xbr, ybr, W, H)

            box_el = ET.SubElement(image_el, "box", {
                "label": label_name,
                "occluded": "0",
                "xtl": f"{xtl:.2f}", "ytl": f"{ytl:.2f}",
                "xbr": f"{xbr:.2f}", "ybr": f"{ybr:.2f}",
                "z_order": "0",
            })
            attr_el = ET.SubElement(box_el, "attribute", {"name": "truncated"})
            attr_el.text = "true" if truncated else "false"
            has_any_object = True

            # stuff-class không có instance thật -> luôn đưa review để người xác nhận
            add_review(img_name, "UNCERTAIN_CLASS",
                       f"{label_name} tách bằng connected-components (không phải instance model) -> cần xác nhận")

    # ---- 7.3 POLYGON cho area/drivable (road) ----
    road_seg_id = next((s["id"] for s in segments_info if s["label_id"] == CITYSCAPES_ROAD_ID), None)
    if road_seg_id is not None:
        road_mask = (seg_map == road_seg_id).astype(np.uint8) * 255
        polygons = mask_to_polygons(road_mask)
        for pts in polygons:
            points_str = ";".join(f"{x:.2f},{y:.2f}" for x, y in pts)
            ET.SubElement(image_el, "polygon", {
                "label": "area/drivable",
                "occluded": "0",
                "points": points_str,
                "z_order": "0",
            })
        if polygons:
            has_any_object = True
        # Model KHÔNG phân biệt được area/alternative -> luôn nhắc reviewer kiểm tra
        add_review(img_name, "UNCERTAIN_SCOPE",
                   "area/alternative KHÔNG được auto-label (model không phân biệt) -> vẽ tay nếu có")
    else:
        add_review(img_name, "UNCERTAIN_BOUNDARY", "Không phát hiện được vùng road nào để tạo area/drivable")

    # ---- 7.4 POLYLINE cho lane marking: KHÔNG auto-generate ----
    # Cityscapes không có nhãn lane marking -> theo nguyên tắc "không đoán",
    # script chủ động BỎ QUA và ghi review để annotator tự vẽ polyline.
    add_review(img_name, "UNCERTAIN_SCOPE",
               "Toàn bộ lane/* (polyline) chưa được auto-label -> cần vẽ tay theo mục 4.2 guideline")

    if not has_any_object:
        add_review(img_name, "UNCERTAIN_SCOPE", "Không có object/area nào được auto-label cho ảnh này")

# -----------------------------------------------------------------------------
# 8. GHI FILE XML (CVAT 1.1 for images) — import trực tiếp vào job CVAT
# -----------------------------------------------------------------------------
xml_path = os.path.join(OUTPUT_DIR, "annotations.xml")
xml_bytes = ET.tostring(root, encoding="utf-8")
pretty_xml = minidom.parseString(xml_bytes).toprettyxml(indent="  ")
with open(xml_path, "w", encoding="utf-8") as f:
    f.write(pretty_xml)

# -----------------------------------------------------------------------------
# 9. GHI REVIEW LOG (đúng format Issue mục 5 guideline) — dùng để tạo
#    Issue/comment trong CVAT hoặc theo dõi ngoài Excel/Sheet
# -----------------------------------------------------------------------------
review_csv_path = os.path.join(OUTPUT_DIR, "review_log.csv")
with open(review_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image", "issue_type", "detail"])
    writer.writeheader()
    writer.writerows(review_rows)

print(f"\nĐã tạo {len(review_rows)} dòng review_log.csv — hãy xử lý các case này trước khi Submit.")

# -----------------------------------------------------------------------------
# 10. ĐÓNG GÓI VÀ TẢI VỀ
# -----------------------------------------------------------------------------
zip_filename = "CVAT_AutoLabel_BBox_Polygon_Polyline.zip"
with zipfile.ZipFile(zip_filename, "w") as zipf:
    zipf.write(xml_path, arcname="annotations.xml")
    zipf.write(review_csv_path, arcname="review_log.csv")

print(f"Xong! File: {zip_filename}")
if IN_COLAB:
    files.download(zip_filename)

# =============================================================================
# GHI CHÚ QUAN TRỌNG (đọc trước khi dùng script này trong pipeline thật)
# =============================================================================
# 1) LABEL THỪA TRONG Labels.md:
#    Job CVAT id=1459 (Labels.md) khai báo thêm road/sidewalk/building/wall/
#    fence/pole/vegetation/terrain/sky/person/traffic_light/traffic_sign —
#    các label này KHÔNG có trong guideline chính thức (chỉ có 19 label ở
#    mục 2 của guideline). Script này CHỦ ĐỘNG KHÔNG sinh annotation cho các
#    label thừa đó để tuân thủ "không tự tạo class, không dùng class ngoài
#    scope". Bạn nên báo lại admin/mentor để dọn lại label set của job trên
#    CVAT nếu 12 label thừa kia không cần dùng.
#
# 2) LANE MARKING (polyline) KHÔNG được auto-label:
#    Model Cityscapes (semantic lẫn panoptic) không có nhãn lane marking.
#    Không có model segmentation phổ biến nào cho ra thẳng 7 class lane/*
#    của bạn (crosswalk/double white/double yellow/road curb/single other/
#    single white/single yellow). Muốn auto-label lane, cần model chuyên
#    biệt (vd: CULane/TuSimple lane-detection) rồi tự map class — ngoài
#    phạm vi model hiện tại. Theo đúng nguyên tắc "không đoán" của guideline,
#    script để trống phần này và ghi review, bạn/annotator vẽ tay.
#
# 3) traffic light / traffic sign LÀ "STUFF CLASS" trong Cityscapes panoptic:
#    Không có instance ID riêng như xe/người, nên script tách bằng
#    connected-components (mỗi cụm pixel liền kề = 1 box). Cách này DỄ SAI
#    khi nhiều biển báo/đèn đứng sát nhau (dính thành 1 box) hoặc bị che
#    một phần (tách thành 2 box của cùng 1 vật). Mọi box loại này được
#    tự động gắn UNCERTAIN_CLASS trong review_log.csv để bạn kiểm tra kỹ.
#
# 4) area/alternative KHÔNG phân biệt được với area/drivable:
#    Cityscapes chỉ có 1 class "road" chung, không có khái niệm
#    drivable vs alternative theo định nghĩa batch của bạn (mục 4.1
#    guideline). Toàn bộ vùng road được gán mặc định là "area/drivable";
#    nếu ảnh có làn phụ/alternative cần phân biệt, PHẢI vẽ tay.
#
# 5) occluded được suy ra bằng heuristic (tỉ lệ pixel mask / diện tích bbox
#    < 0.65), CHỈ mang tính gợi ý — không thay thế đánh giá thị giác thật
#    của annotator. Mọi box occluded=true tự động đều bị đưa vào review_log.
#
# 6) THRESHOLD (SCORE_THRESHOLD, MIN_BOX_AREA_PX, OCCLUDED_FILL_RATIO...) là
#    tham số có thể tinh chỉnh theo chất lượng ảnh thực tế của batch — nên
#    thử trên một vài ảnh mẫu và so sánh với ground truth trước khi chạy
#    hàng loạt, đúng tinh thần "annotation nhất quán và evaluate được".
#
# 7) Workflow đề xuất khi dùng script:
#    export images (CVAT) -> đặt vào thư mục "images/" cạnh script
#    -> chạy script -> import annotations.xml vào job (Upload annotations,
#       format "CVAT 1.1") -> mở review_log.csv, tạo Issue trong CVAT cho
#       từng dòng -> annotator/reviewer edit label & vẽ tay phần lane +
#       area/alternative -> tự review theo checklist mục 6 guideline
#       trước khi chuyển Validation.
# =============================================================================